In [1]:
import os
os.chdir('..')
os.chdir('..')


In [2]:
import timm
import torch
import librosa
from classification.features.mel_spectrogram import AudioPreprocessor

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [17]:
model = timm.create_model(
    "convnext_tiny_hnf.a2h_in1k",
    pretrained=False,
    in_chans=1,
    num_classes=2,
).to(device)

state_dict = torch.load(
    r"D:\Project\Synthetic Speech Recognizer\models\convnext_tiny\best_model.pt",
    map_location=device,
)

model.load_state_dict(state_dict)

model.eval()

C:\Users\VUONG\AppData\Local\Temp\ipykernel_10356\3765282248.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(


ConvNeXt(
  (stem): Sequential(
    (0): Conv2d(1, 96, kernel_size=(4, 4), stride=(4, 4))
    (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True)
  )
  (stages): Sequential(
    (0): ConvNeXtStage(
      (downsample): Identity()
      (blocks): Sequential(
        (0): ConvNeXtBlock(
          (conv_dw): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (norm): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Conv2d(96, 384, kernel_size=(1, 1), stride=(1, 1))
            (act): GELU()
            (drop1): Dropout(p=0.0, inplace=False)
            (norm): Identity()
            (fc2): Conv2d(384, 96, kernel_size=(1, 1), stride=(1, 1))
            (drop2): Dropout(p=0.0, inplace=False)
          )
          (shortcut): Identity()
          (drop_path): Identity()
        )
        (1): ConvNeXtBlock(
          (conv_dw): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)


In [18]:
preprocessor = AudioPreprocessor(
    sr=16000,
    target_length=48000,
    n_fft=1024,
    hop_length=256,
    n_mels=128,
    normalize_wav=False,
)

In [20]:
wav, _ = librosa.load(
    audio_path,
    sr=preprocessor.sr,
    mono=True
)

spec = preprocessor.process(
    wav,
    training=False
)

x = spec.unsqueeze(0).to(device)

In [34]:
def predict_audio(audio_path: str):
    with torch.inference_mode():
        logits = model(x)
        prediction = torch.argmax(
                logits,
                dim=1,
            ).item()

        confidence = logits[
                0,
                prediction
            ].item()

        classes = {
            0: "real",
            1: "fake",
        }

        return {
            "class_id": prediction,
            "class": classes[prediction],
            "confidence": confidence,
        }

In [41]:
audio_path = r"D:\Project\Synthetic Speech Recognizer\datasets\raw\for-norm\for-norm\training\fake\file3.mp3.wav_16k.wav_norm.wav_mono.wav_silence.wav"
result = predict_audio(audio_path)
print(result)

{'class_id': 0, 'class': 'real', 'confidence': 1.7342196702957153}
